In [ ]:
import os
import math
import numpy as np
import pandas as pd

# import cupy as cp
# import cudf

from typing import Iterator, Tuple
from pyspark.sql.types import DoubleType, LongType
from pyspark.sql.functions import col, count, rand, collect_list, explode, struct, count, lit
from pyspark.sql.functions import udf, pandas_udf

In [ ]:
from numba import jit
from numba import cuda

In [ ]:
cuda.is_available()

In [ ]:
np.__config__.show()

In [ ]:
@cuda.jit
def _cuLonToX(lon, out):
    i = cuda.grid(1)
    if i < lon.size:
        out[i] = 6378137.0 * math.radians(lon[i])
        

@pandas_udf(returnType=DoubleType())
def cuLonToX(lon:pd.Series)->pd.Series:
    arr = cp.asarray(lon)
    out = cp.empty(arr.size)
    _cuLonToX.forall(arr.size)(arr,out)
    return pd.Series(out.tolist())

In [ ]:
@pandas_udf(returnType=DoubleType())
def pdLonToX(lon:pd.Series)->pd.Series:
    return 6378137.0 * np.radians(lon)

@pandas_udf(returnType=DoubleType())
def pdLatToY(lat:pd.Series)->pd.Series:
    return 6378137.0 * np.log(np.tan((math.pi * 0.25) + (0.5 * np.radians(lat))))

In [ ]:
%%time

spark\
    .range(1_000_000)\
    .selectExpr("id","-180.0+360.0*rand() lon1","-90.0+180.0*rand() lat1","-180.0+360.0*rand() lon2","-90.0+180.0*rand() lat2")\
    .withColumn("x", pdLonToX("lon1"))\
    .createOrReplaceTempView("v0")

sql("""select avg(x) from v0""").collect()

| Method | Time |
|--------|------|
| GPU cuLonToX | 600 ms |
| GPU pdLonToX | 400 ms |
| CPU pdLonToX | 7.1 s |